# Dataset-level node t-SNE

This is the paper-style node scatter visualization: **one 2-D point = one response-token node**. No GNN is trained here. Every node is represented by the 12-D interpretable structural state extracted directly from reconstructed canonical attention, then all selected test nodes are standardized and fitted by one common t-SNE.

Normal/hallucination labels are not used to construct the 12-D node states. They are used only to color/mark the final scatter. `MAX_NODES=None` uses every response token in the selected samples; set it to e.g. `20000` only if a full t-SNE is too slow.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
from sklearn.preprocessing import StandardScaler

from node_tsne import NodeTSNEVisualizer

DATA_ROOT = Path(
    "/share/home/tm902089733300000/a903202310/lys/data/RAGTruth/"
    "model_traces/llama31_8b/test"
)
OUTPUT_ROOT = REPO_ROOT / "outputs" / "node_tsne" / "test"

MAX_SAMPLES = None  # None = all test samples
MAX_NODES = None    # None = every response token; try 20000 for a quick run
PERPLEXITY = 30.0
RANDOM_STATE = 0


## Fit one common node embedding over the test split

For each sample, `ResearchSample.graph_view(labels)` returns a `[response_tokens, 12]` node-state matrix. The matrices are concatenated across samples first, and t-SNE is fitted once on the pooled matrix. Therefore coordinates are directly comparable across samples.


In [ ]:
viewer = NodeTSNEVisualizer(
    DATA_ROOT,
    device="cpu",
    verify_hashes=False,
    random_state=RANDOM_STATE,
)

result, metadata = viewer.run(
    output_dir=OUTPUT_ROOT,
    max_samples=MAX_SAMPLES,
    max_nodes=MAX_NODES,
    perplexity=PERPLEXITY,
)
metadata


## Inspect whether the separation matches the structural hypothesis

The t-SNE plot is qualitative. The following post-hoc standardized mean differences show which of the 12 structural coordinates differ most between hallucination and correct nodes. Positive means larger for hallucination nodes; negative means smaller. This calculation does not change the embedding.


In [ ]:
features = result["features"]
labels = result["labels"]
feature_names = result["feature_names"]

scaled = StandardScaler().fit_transform(features)
difference = (
    scaled[labels == 1].mean(axis=0)
    - scaled[labels == 0].mean(axis=0)
)
order = np.argsort(np.abs(difference))[::-1]
[(str(feature_names[i]), float(difference[i])) for i in order]


## Saved outputs

`outputs/node_tsne/test/node_tsne.png` is the scatter plot. `node_tsne_coordinates.npz` keeps the 2-D coordinates, original 12-D node states, token labels, sample IDs, response positions, task/data-source metadata and feature names, so later analyses do not need to rerun t-SNE. `metadata.json` records counts and settings.
